In [3]:
import tensorflow as tf
import tkinter as tk
from tkinter import messagebox
import numpy as np
from PIL import Image, ImageDraw

CANVAS_SIZE = 280
DRAW_SIZE = 28
BRUSH_WIDTH = 10

model = tf.keras.models.load_model('mlp_model.keras')

class DigitClassifierApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Clasificador de numeros")
        self.root.resizable(False, False)
        self.previous_point = None
        self.drawing_image = Image.new("L", (CANVAS_SIZE, CANVAS_SIZE), 255)
        self.drawing_context = ImageDraw.Draw(self.drawing_image)
        self.model = model

        tk.Label(root, text="Dibuja un numero (0-9)", font=("Segoe UI", 16, "bold")).pack(pady=(12, 8))
        self.canvas = tk.Canvas(root, width=CANVAS_SIZE, height=CANVAS_SIZE, bg="white",
                                cursor="crosshair", highlightthickness=1,
                                highlightbackground="#7f8c8d")
        self.canvas.pack()
        self.canvas.bind("<ButtonPress-1>", self.start_drawing)
        self.canvas.bind("<B1-Motion>", self.draw)
        self.canvas.bind("<ButtonRelease-1>", self.stop_drawing)

        buttons = tk.Frame(root)
        buttons.pack(pady=12)
        tk.Button(buttons, text="Clasificar dibujo", command=self.classify_drawing,
                  font=("Segoe UI", 11), width=18).grid(row=0, column=0, padx=5)
        tk.Button(buttons, text="Limpiar", command=self.clear_drawing,
                  font=("Segoe UI", 11), width=12).grid(row=0, column=1, padx=5)
        self.result_label = tk.Label(root, text="Resultado: -", font=("Segoe UI", 14), pady=4)
        self.result_label.pack(pady=(0, 12))

    def start_drawing(self, event):
        self.previous_point = (event.x, event.y)
        self.draw(event)

    def draw(self, event):
        if self.previous_point is None:
            return
        x0, y0 = self.previous_point
        x1, y1 = event.x, event.y
        self.canvas.create_line(x0, y0, x1, y1, fill="black", width=BRUSH_WIDTH,
                                capstyle=tk.ROUND, smooth=True)
        self.drawing_context.line((x0, y0, x1, y1), fill=0, width=BRUSH_WIDTH)
        radius = BRUSH_WIDTH // 2
        self.drawing_context.ellipse((x1 - radius, y1 - radius,
                                      x1 + radius, y1 + radius), fill=0)
        self.previous_point = (x1, y1)

    def stop_drawing(self, _event):
        self.previous_point = None

    def clear_drawing(self):
        self.canvas.delete("all")
        self.drawing_image = Image.new("L", (CANVAS_SIZE, CANVAS_SIZE), 255)
        self.drawing_context = ImageDraw.Draw(self.drawing_image)
        self.result_label.config(text="Resultado: -")

    def prepare_image(self):
        resized = self.drawing_image.resize((DRAW_SIZE, DRAW_SIZE), Image.Resampling.LANCZOS)
        digit = 255 - np.asarray(resized, dtype=np.uint8)
        coordinates = np.column_stack(np.where(digit > 20))
        if coordinates.size == 0:
            raise ValueError("Dibuja un numero antes de clasificarlo.")
        y_min, x_min = coordinates.min(axis=0)
        y_max, x_max = coordinates.max(axis=0)
        cropped = digit[y_min:y_max + 1, x_min:x_max + 1]
        height, width = cropped.shape
        scale = 20.0 / max(height, width)
        new_width = max(1, round(width * scale))
        new_height = max(1, round(height * scale))
        cropped_image = Image.fromarray(cropped, mode="L").resize(
            (new_width, new_height), Image.Resampling.LANCZOS)
        centered = Image.new("L", (DRAW_SIZE, DRAW_SIZE), 0)
        centered.paste(cropped_image, ((DRAW_SIZE - new_width) // 2,
                                       (DRAW_SIZE - new_height) // 2))
        return np.asarray(centered, dtype=np.float32).reshape(DRAW_SIZE, DRAW_SIZE) / 255.0

    def classify_drawing(self):
        try:
            image = self.prepare_image()
            prediction = self.model.predict(image.reshape(1, DRAW_SIZE, DRAW_SIZE), verbose=0)
            digit = int(np.argmax(prediction[0]))
            confidence = float(prediction[0][digit]) * 100
            self.result_label.config(text=f"Resultado: {digit} ({confidence:.1f}% de confianza)")
        except ValueError as error:
            messagebox.showinfo("Clasificador", str(error))
        except Exception as error:
            messagebox.showerror("Error al clasificar", str(error))

root = tk.Tk()
app = DigitClassifierApp(root)
root.mainloop()
